In [15]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [37]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12288, hidden_size_1=256, hidden_size_2=128, hidden_size_3=64, speed_size=1,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2 + speed_size, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, image, speed):
        out = self.fc1(image)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = torch.cat((out, speed), dim=1)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [17]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_road_version_4.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_speed_version_4.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_for_lstm_wheel_4.csv')

In [18]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [19]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

13037
13037
13037


In [33]:
dataset = TensorDataset(X_train, S_train, y_tensor)
train_loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [43]:
model = FeedforwardNet().cuda()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

# Обучение модели
num_epochs = 40
for epoch in range(num_epochs):
    for X, S, y in train_loader:
        # Прямое прохождение
        output = model(X, S)
        loss = criterion(output, y)

        # Обратное распространение и обновление весов
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/40], Loss: 0.015117340
Epoch [2/40], Loss: 0.017292934
Epoch [3/40], Loss: 0.016758217
Epoch [4/40], Loss: 0.014939489
Epoch [5/40], Loss: 0.013550164
Epoch [6/40], Loss: 0.012710687
Epoch [7/40], Loss: 0.011884010
Epoch [8/40], Loss: 0.011071426
Epoch [9/40], Loss: 0.009732582
Epoch [10/40], Loss: 0.008041019
Epoch [11/40], Loss: 0.006303973
Epoch [12/40], Loss: 0.004670547
Epoch [13/40], Loss: 0.003247635
Epoch [14/40], Loss: 0.002201832
Epoch [15/40], Loss: 0.001493623
Epoch [16/40], Loss: 0.001031312
Epoch [17/40], Loss: 0.000728393
Epoch [18/40], Loss: 0.000524284
Epoch [19/40], Loss: 0.000384894
Epoch [20/40], Loss: 0.000289481
Epoch [21/40], Loss: 0.000223297
Epoch [22/40], Loss: 0.000177779
Epoch [23/40], Loss: 0.000145670
Epoch [24/40], Loss: 0.000123621
Epoch [25/40], Loss: 0.000109504
Epoch [26/40], Loss: 0.000100501
Epoch [27/40], Loss: 0.000095098
Epoch [28/40], Loss: 0.000091360
Epoch [29/40], Loss: 0.000088577
Epoch [30/40], Loss: 0.000085891
Epoch [31/40], Loss

In [44]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.000001)

num_epochs = 50
for epoch in range(num_epochs):
    for X, S, y in train_loader:
        # Прямое прохождение
        output = model(X, S)
        loss = criterion(output, y)

        # Обратное распространение и обновление весов
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/50], Loss: 0.000205684
Epoch [2/50], Loss: 0.000157397
Epoch [3/50], Loss: 0.000126500
Epoch [4/50], Loss: 0.000106056
Epoch [5/50], Loss: 0.000093024
Epoch [6/50], Loss: 0.000084887
Epoch [7/50], Loss: 0.000079563
Epoch [8/50], Loss: 0.000075709
Epoch [9/50], Loss: 0.000072616
Epoch [10/50], Loss: 0.000070030
Epoch [11/50], Loss: 0.000067733
Epoch [12/50], Loss: 0.000065705
Epoch [13/50], Loss: 0.000063647
Epoch [14/50], Loss: 0.000061796
Epoch [15/50], Loss: 0.000060001
Epoch [16/50], Loss: 0.000058310
Epoch [17/50], Loss: 0.000056674
Epoch [18/50], Loss: 0.000055049
Epoch [19/50], Loss: 0.000053583
Epoch [20/50], Loss: 0.000052143
Epoch [21/50], Loss: 0.000050767
Epoch [22/50], Loss: 0.000049442
Epoch [23/50], Loss: 0.000048046
Epoch [24/50], Loss: 0.000046814
Epoch [25/50], Loss: 0.000045672
Epoch [26/50], Loss: 0.000044420
Epoch [27/50], Loss: 0.000043244
Epoch [28/50], Loss: 0.000042106
Epoch [29/50], Loss: 0.000040883
Epoch [30/50], Loss: 0.000040059
Epoch [31/50], Loss

In [45]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_version_2.pth')